### Tools

In [15]:
import os
from langchain.chat_models import init_chat_model
model=init_chat_model('groq:qwen/qwen3-32b')

response=model.invoke("Hello How are you")

response.content

'<think>\nOkay, the user said, "Hello How are you." I need to respond appropriately. First, I should acknowledge their greeting and offer a friendly reply. Since they asked how I am, I should mention that I\'m functioning well. Then, I should ask how they are to keep the conversation going. I should keep it simple and open-ended. Let me make sure the tone is warm and welcoming. Also, I need to check for any typos or errors in the response. Alright, that should cover it.\n</think>\n\nHello! I\'m doing well, thank you for asking. How are you today? 😊'

In [19]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"Its sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

response=model_with_tools.invoke('WHat`s the weather like in Jamnagar')

print(response)
for tool_call in response.tool_calls:
    # view tool call made by modal
    print(f"Tool : {tool_call['name']}")
    print(f"Args : {tool_call['args']}")



content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Jamnagar. Let me check if I have the right tool for that. The available function is get_weather, which requires a location parameter. Jamnagar is a city in India, so I should use that function. I need to make sure the location is specified correctly. The parameters should be {"location": "Jamnagar"}. Alright, I\'ll format the tool call accordingly.\n', 'tool_calls': [{'id': 'b4vgtc5sq', 'function': {'arguments': '{"location":"Jamnagar"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 156, 'total_tokens': 271, 'completion_time': 0.169258405, 'completion_tokens_details': {'reasoning_tokens': 89}, 'prompt_time': 0.0061076, 'prompt_tokens_details': None, 'queue_time': 0.15692312, 'total_time': 0.175366005}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_rea

### Tool Execution Loops

In [18]:
# step 1 model generate tool calls

messages=[{"role":"user","content":"What`s the weather in Jamnagar?"}]

ai_msg=model_with_tools.invoke(messages);

messages.append(ai_msg)


# step 2 Execute tools and collect results

for tool_call in ai_msg.tool_calls:
    # Execure the tool with the generated arguments
    tool_result= get_weather.invoke(tool_call)
    messages.append(tool_result);

#step 3 : pass result back to model for final resonse
final_response=model_with_tools.invoke(messages)
print(final_response.text)

# "the current weather in jamnagar is 32c and sunny"

messages


The weather in Jamnagar is currently sunny.


[{'role': 'user', 'content': 'What`s the weather in Jamnagar?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Jamnagar. Let me check the tools provided. There\'s a function called get_weather that takes a location parameter. Since Jamnagar is a specific location, I should call that function with "Jamnagar" as the argument. I need to make sure the JSON is correctly formatted with the location parameter. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': 'a45t762bs', 'function': {'arguments': '{"location":"Jamnagar"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 155, 'total_tokens': 262, 'completion_time': 0.158924473, 'completion_tokens_details': {'reasoning_tokens': 81}, 'prompt_time': 0.007364213, 'prompt_tokens_details': None, 'queue_time': 0.055508716, 'total_time': 0.166288686}, 'model_name': 'qwen/qwen3-32b', 'system_fin